# Store Sales Forecasting — EDA and Feature Engineering

This notebook preserves the original exploratory work. Required CSV files must be placed in the project `data/` directory; see `data/README.md`. The daily calendar is completed before lag and rolling features are calculated so row shifts align with calendar days.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# **Prepare the DataFrame**

In [ ]:
import pandas as pd
#import os

#os.listdir('/kaggle/input')

train = pd.read_csv(DATA_DIR / 'train.csv')
train

In [ ]:
train.shape

In [ ]:
# Convert the date column to datetime
train["date"] = pd.to_datetime(train["date"])

# Check data types
print(train.dtypes)

# Check dimensions
print("\nDimensions:", train.shape)

# Check the date range
print("\nDate range:", train["date"].min(), "to", train["date"].max())

In [ ]:
# **stores.csv file**

# **stores.csv**

In [ ]:
stores = pd.read_csv(DATA_DIR / 'stores.csv')
stores

In [ ]:
print(f"Dimensions : {stores.shape}")

print(f"-----Information----- : ")
print(stores.info())

print("-----Missing values----- : ")
print(stores.isna().sum())

print("-----Duplicate values----- : ")
print(stores.duplicated().sum())

print("-----Number of stores----- : ")
print(stores["store_nbr"].nunique())

## **test.csv**

In [ ]:
test = pd.read_csv(DATA_DIR / 'test.csv')
test

In [ ]:
print("----- Dimensions of the test set -----")
print(test.shape)

print("----- Information -----")
test.info()

print("----- Missing values -----")
print(test.isna().sum())

print("----- Duplicate values -----")
print(test.duplicated().sum())

test["date"] = pd.to_datetime(test["date"])
print("----- Date range -----")
print("Start date:", test["date"].min())
print("End date:", test["date"].max())


# **oil.csv**

In [ ]:
oil = pd.read_csv(DATA_DIR / 'oil.csv')
oil

In [ ]:
print("----- Row and column count -----\n", oil.shape)

print("----- Information -----\n")
print(oil.info())

print("----- Missing values -----")
print(oil.isna().sum())

print("----- Duplicate values -----")
print(oil.duplicated().sum())

oil['date'] = pd.to_datetime(oil['date'])

print("----- Date range -----")
print("Start date:", oil['date'].min())
print("End date:", oil['date'].max())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(oil["date"], oil["dcoilwtico"])

plt.title("Oil price over time (2013–2017)")
plt.xlabel("Data")
plt.ylabel("Oil price (USD)")
plt.grid(True)

plt.show()

# **holidays_events.csv**

In [ ]:
holidays = pd.read_csv(DATA_DIR / "holidays_events.csv")

display(holidays)

In [ ]:
print("Dimensions:", holidays.shape)

print("\nInformation:")
holidays.info()

print("\nMissing values:")
print(holidays.isna().sum())

print("\nDuplicate records:")
print(holidays.duplicated().sum())

holidays["date"] = pd.to_datetime(holidays["date"])

print("\nStart date:", holidays["date"].min())
print("End date:", holidays["date"].max())

In [ ]:
print("Event types:")
print(holidays["type"].value_counts()) # count the frequency of unique values in a column (Series) or DataFrame

print("\nLocale types:")
print(holidays["locale"].value_counts())

print("\nTransferred events:")
print(holidays["transferred"].value_counts())

print("\nSample events:")
display(holidays.head(10))

**Check for dates with multiple events**

In [ ]:
# Count how many events occur on each date
events_per_date = holidays.groupby("date").size()

# Show dates with more than one event
dates_with_multiple_events = events_per_date[events_per_date > 1]

print("Number of dates with multiple events:",
      len(dates_with_multiple_events))

print("\n_examples:")
print(dates_with_multiple_events)

In [ ]:
# Inspect the events on one of these dates
if len(dates_with_multiple_events) > 0:
    example_date = dates_with_multiple_events.index[0]

    display(
        holidays[holidays["date"] == example_date]
    )

**Merge store data into the training set**

In [ ]:
train_merged = train.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

print("Original dimensions:", train.shape)
print("Dimensions after merge:", train_merged.shape)

print("\nMissing store attributes:")
print(
    train_merged[
        ["city", "state", "type", "cluster"]
    ].isna().sum()
)

display(train_merged.head())

**Merge store data into the test set**

In [ ]:
test_merged = test.merge(
    stores,
    on = "store_nbr",
    how = "left",
    validate = "many_to_one"
)

print("Original dimensions:", test.shape)
print("Dimensions after merge:", test_merged.shape)

print("\nMissing values:")
print(test_merged[["city", "state", "type", "cluster"]].isna().sum())

display(test_merged.head())

**Remove columns from the merged training set**

In [ ]:
train_merged = train_merged.drop(
    columns=["dcoilwtico", "dcoilwtico_x", "dcoilwtico_y"],
    errors="ignore"
)

test_merged = test_merged.drop(
    columns=["dcoilwtico", "dcoilwtico_x", "dcoilwtico_y"],
    errors="ignore"
)

In [ ]:
print(train_merged.columns.tolist())
print(test_merged.columns.tolist())

**Merge oil prices**

In [ ]:
train_merged["date"] = pd.to_datetime(train_merged["date"])
test_merged["date"] = pd.to_datetime(test_merged["date"])
oil["date"] = pd.to_datetime(oil["date"])


train_merged = train_merged.merge(
    oil,
    on="date",
    how="left",
    validate="many_to_one"
)

test_merged = test_merged.merge(
    oil,
    on="date",
    how="left",
    validate="many_to_one"
)

print("Training:", train_merged.shape)
print("Test:", test_merged.shape)

print("\nMissing oil prices:")
print("Training:", train_merged["dcoilwtico"].isna().sum())
print("Test:", test_merged["dcoilwtico"].isna().sum())

In [ ]:
print("Training date range:")
print(train_merged["date"].min(), train_merged["date"].max())

print("\nTest date range:")
print(test_merged["date"].min(), test_merged["date"].max())

print("\nOil date range:")
print(oil["date"].min(), oil["date"].max())

print("\nTest dates without oil prices:")
print(
    test_merged.loc[
        test_merged["dcoilwtico"].isna(),
        "date"
    ].value_counts().sort_index()
)

print("\nSample test dates:")
print(test_merged["date"].drop_duplicates().sort_values().head())

In [ ]:
print("TEST ORIGINAL:")
print(test.shape)
print(test["date"].min(), test["date"].max())
display(test.head())
display(test.tail())

print("\nMERGED TEST SET:")
print(test_merged.shape)
print(test_merged["date"].min(), test_merged["date"].max())
display(test_merged.head())

In [ ]:
display(
    oil[oil["date"].between("2017-08-16", "2017-08-31")]
)

In [ ]:
# Reload directly from the original file
test = pd.read_csv(DATA_DIR / "test.csv")

# Check dates before making any changes
print("Dimensions:", test.shape)
print("First dates:")
print(test["date"].head())

print("\nLast dates:")
print(test["date"].tail())

print("\nDate range:")
print(test["date"].min(), test["date"].max())

**Rebuild the merged test set from scratch**

In [ ]:
# Convert dates
test["date"] = pd.to_datetime(test["date"])
oil["date"] = pd.to_datetime(oil["date"])

# Integrar stores
test_merged = test.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

# Merge oil prices
test_merged = test_merged.merge(
    oil,
    on="date",
    how="left",
    validate="many_to_one"
)

# Check the result
print("Dimensions:", test_merged.shape)
print("Date range:", test_merged["date"].min(),
      "to", test_merged["date"].max())
print("Missing values:\n", test_merged.isna().sum())

**Handle missing oil prices**

In [ ]:
display(test_merged)
print("Missing values in column 'dcoilwtico' in merged test set: ")
print(test_merged['dcoilwtico'].isna().sum())

In [ ]:
display(train_merged)
print("Missing values in column 'dcoilwtico' in merged training set: ")
print(train_merged['dcoilwtico'].isna().sum())

In [ ]:
# Sort by date
train_merged = train_merged.sort_values('date')
test_merged = test_merged.sort_values('date')

# Fill missing values
train_merged['dcoilwtico'] = ( train_merged['dcoilwtico'].ffill() ) # fill missing values (NaN or None) with the most recent valid value
test_merged['dcoilwtico'] = ( test_merged['dcoilwtico'].ffill() )

# Check
print("Missing values in column 'dcoilwtico' in merged training set: ")
print(train_merged['dcoilwtico'].isna().sum())
'''Why are 1,782 values still missing in the training set?
ffill() uses the most recent valid value. If the first dates have no quote, there is no earlier value to propagate.'''

print("Missing values in 'dcoilwtico' in the merged test set: ")
print(test_merged["dcoilwtico"].isna().sum())

In [ ]:
# Investigate the 1,782 missing oil-price values in the merged training set.
# They all belong to 2013-01-01.
# Forward fill cannot fill the first date in the history because no earlier
# oil price is available.


In [ ]:
train_merged

In [ ]:
# Fill missing or null values (NaN) with the next valid value in the dataset.
train_merged['dcoilwtico'] = ( train_merged['dcoilwtico'].bfill())
print("Missing values in column 'dcoilwtico' in merged training set: ")
print(train_merged['dcoilwtico'].isna().sum())

train_merged

We use `ffill()` and `bfill()` because oil prices are time-series data that vary over time. The choice should reflect the nature of the variable.

### Why not use the mean or median?

| Method | How it works | Limitation in this case |
|---|---|---|
| Mean | Replaces a value with the average of all prices | Ignores changes over time |
| Median | Replaces a value with the middle price | Also ignores temporal order |
| `ffill()` | Repeats the last known price | Preserves forward continuity |
| `bfill()` | Uses the next known price | Can fill missing values at the beginning |

For example, `ffill()` fills a missing value with the previous price, while `bfill()` can fill a leading gap using a later observation. This is a reasonable strategy for this series, but it is not always the best choice. We should document it as an imputation decision and validate it before using the data for modeling.

**Understand the holiday data structure**

In [ ]:
print(holidays.head(10))

print("\nHoliday types:")
print(holidays["type"].value_counts())

print("\nLocale scope:")
print(holidays["locale"].value_counts())

print("\nRegions:")
print(holidays["locale_name"].value_counts())

### **Structural Analysis of `holidays_events.csv`**

The dataset contains six event types: `Holiday` (221 records), `Event` (56), `Additional` (51), `Transfer` (12), `Bridge` (5), and `Work Day` (5). `Work Day` records represent working days used to compensate for bridges or transferred holidays.

There are three geographic scopes: `National` (174 records, generally associated with Ecuador), `Local` (152 records, tied to specific cities), and `Regional` (24 records, tied to provinces).

Twelve events have `transferred = True`. Their original date may no longer behave like a holiday, while the replacement date may do so. This transfer logic must be handled carefully when joining the calendar.

# **Prepare the joins**

In [ ]:
print("Store cities: ")
print(sorted(stores['city'])) # sort the elements of an iterable and return a new list without changing the original object

print("Store states: ")
print(sorted(stores['state'].unique()))

print("Holidays tranferiveis: ")
print(sorted(holidays['transferred'] == True))

print("Event types: ")
print(sorted(holidays['type'].unique()))

### **Store Geography and Join Keys**

The dataset contains 54 stores across multiple cities. Thuto has the largest concentration, followed by major cities such as Guayaquil, Cuenca, and Santo Suningo; some cities have only one store. Stores span 16 provinces, including Pichincha and Guayas.

Most holiday records are not transferred. Exactly 12 have `transferred = True`; for those, the original date and the replacement `Transfer` event may have different commercial effects. Event types in the source include `Additional`, `Bridge`, `Event`, `Holiday`, `Transfer`, and `Work Day`.

These geographic levels determine the join keys: store number for store attributes, and date plus state or city for holidays.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the chart style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Chart 1: Number of Stores by State
sns.countplot(
    data=stores,
    y="state",
    order=stores["state"].value_counts().index,
    ax=axes[0],
    palette="viridis"
)
axes[0].set_title("Store Distribution by State", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Number of Stores")
axes[0].set_ylabel("State")

# Chart 2: Holiday and Event Types
sns.countplot(
    data=holidays,
    x="type",
    order=holidays["type"].value_counts().index,
    ax=axes[1],
    palette="magma"
)
axes[1].set_title("Frequency of Holiday/Event Types", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Holiday Type")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter transferred holidays only
transferred_holidays = holidays[holidays['transferred'] == True]

# Create the figure
plt.figure(figsize=(12, 6))

# Create a bar chart showing where these holidays occurred
sns.countplot(
    data=transferred_holidays,
    y='locale_name',
    order=transferred_holidays['locale_name'].value_counts().index,
    hue='locale',
    palette='Set2'
)

plt.title('Cities, States, or National Scope of Transferred Holidays (transferred = True)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Transferred Holidays')
plt.ylabel('Locale (locale_name)')
plt.legend(title='Locale scope')
plt.tight_layout()
plt.show()

### **Interpreting the Exploratory Charts**

The store-by-state chart shows that stores are concentrated in Pichincha (Thuto) and Guayas (Guayaquil), while several provinces have only one or two stores. This distribution matters when comparing regional holiday effects.

The event-type chart shows `Holiday` as the most frequent category; `Additional` and `Event` also occur, while `Work Day` and `Bridge` are less common.

The transferred-holiday chart shows that 10 of the 12 transferred records are national, with one local case in Guayaquil and one regional case in Manabi. A transferred holiday should be associated with the correct replacement date rather than automatically treated as active on its original date.

# **Prepare the holiday data**

In [ ]:
# Convert dates
holidays["date"] = pd.to_datetime(holidays["date"])

#remove whitespace (or specified unwanted characters) from the start and end of a string.
stores['city'] = stores['city'].str.strip() # # Remove spaces and standardize names
stores['state'] = stores['state'].str.strip()


holidays["locale_name"] = holidays["locale_name"].str.strip()

# Check data types
print(holidays.dtypes)

# Check transferred holidays
print("\nTransferred holidays:")
print(holidays.loc[
    holidays["transferred"] == True,
    ["date", "type", "locale_name", "description", "transferred"]
])




### **Preparing Holiday Data for Joins**

This section inspects holiday categories and scopes, standardizes city names, and filters national, regional, and local records for integration. It uses `.loc[]` to select records by conditions and checks for multiple events that could duplicate sales rows during a join.

Before merging, consolidate duplicate date-and-scope keys and validate the join cardinality. A left join preserves the sales rows; `validate="many_to_one"` helps catch duplicate matches in the holiday table.

# **Investigate transferred holidays**

In [ ]:
holidays

In [ ]:
print("RECORDS MARKED AS TRANSFERRED")
print(holidays.loc[
    holidays["transferred"] == True,
    ["date","locale","locale_name","description"]
].to_string(index=False)
      )

print("\nTransferred holidays")
print(holidays.loc[
    holidays["transferred"] == True,
    ["date","locale", "locale_name", "description", "transferred"]
].to_string(index=False)
    )

### **Investigating Holiday Transfers**

The next cells compare records marked `transferred = True` (the original dates) with records of type `Transfer` (replacement dates). Matching only by location can produce several candidates, so the exploratory mapping also considers year and event-description keywords.

This is a diagnostic approximation, not a fully validated mapping rule. Review the resulting pairs before using transfer status to create a model feature.

In [ ]:
# Filter transferred holidays (original date, when the day off does not apply)
origins = holidays[holidays['transferred'] == True][['date', 'locale_name', 'description']]

# Filter records for the actual replacement dates (when the day off applies)
destinations = holidays[holidays['type'] == 'Transfer'][['date', 'locale_name', 'description']]

# Approximate the original-to-replacement mapping by locale and description
mapping = origins.merge(
    destinations,
    on='locale_name',
    suffixes=('_original_sem_folga', '_destino_com_folga')
)

# Display the transfer mapping to inspect the date changes
print("TRANSFERRED HOLIDAY MAPPING (Original vs Replacement):")
display(mapping[[
    'locale_name',
    'date_original_sem_folga',
    'description_original_sem_folga',
    'date_destino_com_folga',
    'description_destino_com_folga'
]])

In [ ]:
# 1. Identify the original dates (when the holiday was canceled or moved)
origins = holidays[holidays['transferred'] == True].copy()
origins['year'] = origins['date'].dt.year

# 2. Identify the replacement dates (records of type 'Transfer')
destinations = holidays[holidays['type'] == 'Transfer'].copy()
destinations['year'] = destinations['date'].dt.year

# 3. Join strictly by locale (locale_name) and year
precise_mapping = origins.merge(
    destinations,
    on=['locale_name', 'year'],
    suffixes=('_original_DIA_UTIL_com_compras', '_destino_DIA_DE_FOLGA')
)

# Display the cleaned final result
print("PRECISE TRANSFER MAPPING (Same Year):")
display(precise_mapping[[
    'locale_name',
    'date_original_workday_with_sales',
    'description_original_workday_with_sales',
    'date_replacement_day_off',
    'description_replacement_day_off'
]])

In [ ]:
# 1. Identify the original dates and normalize the key if needed
origins = holidays[holidays['transferred'] == True].copy()
origins['year'] = origins['date'].dt.year

# 2. Identify the actual replacement dates (type 'Transfer')
destinations = holidays[holidays['type'] == 'Transfer'].copy()
destinations['year'] = destinations['date'].dt.year

# 3. Map replacement dates using description keywords (e.g., Guayaquil, Pichincha, Independencia)
final_mappings = []

for idx, orig in origins.iterrows():
    # Find the matching transfer record for the same year and locale
    year_locale_condition = (destinations['year'] == orig['year']) & (destinations['locale_name'] == orig['locale_name'])
    candidates = destinations[year_locale_condition]

    # If there is more than one candidate (as for national holidays), compare description keywords
    if len(candidates) > 1:
        keywords = [w for w in orig['description'].split() if len(w) > 4 and w not in ['Ecuador', 'de', 'del', 'para']]
        for _, cand in candidates.iterrows():
            if any(pc.lower() in cand['description'].lower() for pc in keywords):
                final_mappings.append({
                    'locale_name': orig['locale_name'],
                    'Original_Date_Without_Holiday': orig['date'].strftime('%Y-%m-%d'),
                    'Original_Holiday': orig['description'],
                    'Replacement_Date': cand['date'].strftime('%Y-%m-%d'),
                    'Transferred_Holiday': cand['description']
                })
                break
    elif len(candidates) == 1:
        cand = candidates.iloc[0]
        final_mappings.append({
            'locale_name': orig['locale_name'],
            'Original_Date_Without_Holiday': orig['date'].strftime('%Y-%m-%d'),
            'Original_Holiday': orig['description'],
            'Replacement_Date': cand['date'].strftime('%Y-%m-%d'),
            'Transferred_Holiday': cand['description']
        })

df_mapeado = pd.DataFrame(final_mappings)
print("MAPPING OF THE 12 TRANSFERRED HOLIDAYS (ONE-TO-ONE):")
display(df_mapeado)

# **Create an auxiliary table**

In [ ]:
holidays

In [ ]:
# Check records by date and locale scope
print(holidays.groupby(["date", "locale"]).size().sort_values(ascending=False).head(15))

### **Understanding the Grouping Syntax**

The expression below counts events for each date and locale, sorts the counts in descending order, and displays the 15 largest groups:

```python
holidays.groupby(["date", "locale"]).size().sort_values(ascending=False).head(15)
```

- `.groupby(["date", "locale"])` creates one group for each date-and-scope pair.
- `.size()` counts rows in each group.
- `.sort_values(ascending=False)` places the largest counts first.
- `.head(15)` limits the displayed result to 15 rows.

### **Multiple Events on the Same Date**

The grouping above identifies date-and-locale combinations with more than one event. The largest observed groups have a count of 2, meaning two events share the same date and geographic scope.

This matters because joining an unaggregated holiday table to sales by date and locale can duplicate sales rows. Consolidate or otherwise resolve duplicate keys before the join, and validate the intended cardinality.

# **National holiday table**

In [ ]:
print("1. Unique values (clean list):")
print(holidays["type"].unique())

print("\n2. Frequency by type (record count):")
print(holidays["type"].value_counts())

In [ ]:
national_holidays = holidays[
    (holidays["locale"] == "National") &
    (holidays["transferred"] == False) &
    (holidays["type"] != "Work Day")
].copy()
national_holidays
#national_holidays = national_holidays.groupby("date").size()
#national_holidays
#national_holidays["national_holiday"] = 1

#print(national_holidays.head())
#print("Total dates:", len(national_holidays))

In [ ]:
national_holidays = national_holidays.groupby("date").size().reset_index(name="national_holiday")
national_holidays
#national_holidays["national_holiday"] = 1

#print(national_holidays.head())
#print("Total dates:", len(national_holidays))

In [ ]:
national_holidays["national_holiday"] = 1

print(national_holidays.head())
print("Total dates:", len(national_holidays))

In [ ]:
import pandas as pd

# Define the requested target dates
target_dates = ['2012-08-10', '2012-10-12', '2012-11-02', '2012-11-03', '2012-12-21']

# Filter the original holiday table to these dates only
filtered_holidays = holidays[holidays['date'].dt.strftime('%Y-%m-%d').isin(target_dates)]

# Group by date, count events, and combine their descriptions
tabela_resumo = filtered_holidays.groupby('date').agg(
    event_count=('description', 'size'),
    what_are_the_events=('description', lambda x: ', '.join(x))
).reset_index()

# Format dates for display (YYYY-MM-DD)
tabela_resumo['date'] = tabela_resumo['date'].dt.strftime('%Y-%m-%d')

# Display the consolidated table
display(tabela_resumo)

In [ ]:
national_holidays = holidays[
    (holidays["locale"] == "National") &
    (holidays["transferred"] == False) &
    (holidays["type"] != "Work Day")
].copy()

national_holidays = (
    national_holidays.groupby("date")
    .size()
    .reset_index(name="national_holiday")
)

national_holidays["national_holiday"] = 1

print(national_holidays.head())
print("Total dates:", len(national_holidays))

### **Preparing National Holidays**

National holidays apply across all stores. This section filters national records and consolidates events by date so that each date has at most one row in the auxiliary table. The resulting table can then be joined to sales by date without multiplying rows.

# **Create the regional holiday table**

In [ ]:
regional_holidays = holidays[
    (holidays["locale"] == "Regional") &
    (holidays["transferred"] == False) &
    (holidays["type"] != "Work Day")
].copy()

regional_holidays = (
    regional_holidays[["date", "locale_name"]]
    .drop_duplicates()
)

regional_holidays["regional_holiday"] = 1

print(regional_holidays.head(10))
print("Total records:", len(regional_holidays))

# **Merge holiday data with store data**

In [ ]:
# Remove columns left by previous runs to avoid errors when rerunning this cell
train_merged = train_merged.drop(columns=["regional_holiday", "locale_name"], errors="ignore")
test_merged = test_merged.drop(columns=["regional_holiday", "locale_name"], errors="ignore")

# Merge regional holidays into the training set
train_merged = train_merged.merge(
    regional_holidays,
    left_on=["date", "state"],
    right_on=["date", "locale_name"],
    how="left",
    validate="many_to_one"
)

# Merge regional holidays into the test set
test_merged = test_merged.merge(
    regional_holidays,
    left_on=["date", "state"],
    right_on=["date", "locale_name"],
    how="left",
    validate="many_to_one"
)

# Drop the temporary join column
train_merged = train_merged.drop(columns=["locale_name"], errors="ignore")
test_merged = test_merged.drop(columns=["locale_name"], errors="ignore")

# Fill null values with 0 and convert to integers
train_merged["regional_holiday"] = (
    train_merged["regional_holiday"].fillna(0).astype(int)
)

test_merged["regional_holiday"] = (
    test_merged["regional_holiday"].fillna(0).astype(int)
)

print("--------------------------------------------------")
print("Regional holidays in training set:")
print(train_merged["regional_holiday"].value_counts())
print("\nRegional holidays in the test set:")
print(test_merged["regional_holiday"].value_counts())
print("--------------------------------------------------")

### **Integrating Regional Holidays**

The regional holiday table is joined to training and test rows using date and store state. The `regional_holiday` indicator is 1 when a matching regional holiday exists and 0 otherwise.

The notebook reports 1,023 matching training rows and 28,512 test rows with no regional-holiday matches for the test period (August 16–31, 2017). After a left join, missing indicators are filled with 0 and converted to integers.

Before rerunning the merge, remove any previously created indicator and temporary locale columns to avoid duplicate `_x`/`_y` columns. The join uses `validate="many_to_one"` so duplicate date-and-state keys in the holiday table raise an error instead of multiplying sales rows.

In [ ]:
# 1. Remove unwanted duplicate columns (_x, _y) created by repeated runs
import pandas as pd

# Identify and drop columns with _x or _y suffixes from the training and test sets
train_cleanup_columns = [col for col in train_merged.columns if col.endswith('_x') or col.endswith('_y')]
test_cleanup_columns = [col for col in test_merged.columns if col.endswith('_x') or col.endswith('_y')]

train_merged = train_merged.drop(columns=train_cleanup_columns, errors='ignore')
test_merged = test_merged.drop(columns=test_cleanup_columns, errors='ignore')

# Ensure the clean regional_holiday column is preserved
print("Final columns in merged training set:", train_merged.columns.tolist())
print("Final columns in merged test set:", test_merged.columns.tolist())

In [ ]:
# 2. Plot table growth and data coverage
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Number of columns after data enrichment
stages = ['Original', '+ Stores', '+ Oil', '+ Regional Holidays']
train_columns = [6, 10, 11, 12] # Actual number of columns at each stage

sns.barplot(x=stages, y=train_columns, ax=axes[0], palette="Blues_d", hue=stages, legend=False)
axes[0].set_title("Number of Features (Columns) in Training Set", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Number of Columns")
for i, v in enumerate(train_columns):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Chart 2: Regional-holiday versus regular-day share in the training set
labels = ['Regular Days (0)', 'Regional Holidays (1)']
sizes = train_merged['regional_holiday'].value_counts().values

axes[1].pie(sizes, labels=labels, autopct='%1.3f%%', startangle=90, colors=['#abcdef', '#ff9999'], explode=(0, 0.1))
axes[1].set_title("Share of Regional Holidays in Training Set", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Create a comparison chart of column counts in the training and test tables
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Column-count progression data
column_evolution_data = pd.DataFrame({
    'Stage': ['Original', '+ Stores', '+ Oil', '+ Regional Holidays'] * 2,
    'Dataset': ['Training'] * 4 + ['Test'] * 4,
    'Number of Columns': [6, 10, 11, 12,  # Training
                             5, 9, 10, 11]    # Test
})

# Configure the chart theme
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

# Create a grouped bar chart
ax = sns.barplot(
    data=column_evolution_data,
    x='Stage',
    y='Number of Columns',
    hue='Dataset',
    palette=['#1f77b4', '#aec7e8']
)

# Configure the title and labels
plt.title("Comparison of Feature (Column) Counts: Training vs Test", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Merge Stage", fontsize=12)
plt.ylabel("Number of Columns", fontsize=12)
plt.ylim(0, 14)

# Add values above each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(
            p.get_x() + p.get_width() / 2.0,
            height + 0.2,
            f'{int(height)}',
            ha='center',
            fontweight='bold',
            fontsize=10
        )

plt.tight_layout()
plt.show()

### **Understanding the Merged Tables**

Repeated merges can produce `regional_holiday_x` and `regional_holiday_y`: the left table already contains the column, so Pandas adds suffixes to distinguish both copies. The new right-side indicator is missing on dates without a matching holiday. Remove stale columns before rerunning the merge, then retain one clean indicator.

The sales tables are enriched with store attributes from `stores.csv`, daily oil prices from `oil.csv`, and regional holiday indicators from `holidays_events.csv`. The same feature definitions should be applied consistently to training and test data.

In [ ]:
train_merged

In [ ]:
test_merged

# **Local holidays**

In [ ]:
local_holidays = holidays[
    (holidays["locale"] == "Local") &
    (holidays["transferred"] == False) &
    (holidays["type"] != "Work Day")
].copy()

local_holidays = (
    local_holidays[["date", "locale_name"]]
    .drop_duplicates()
)

local_holidays["local_holiday"] = 1

print(local_holidays.head(10))
print("Total records:", len(local_holidays))

'''There are 147 date-and-city combinations with local holidays.'''

# Merge local holidays by matching `locale_name` to the store's `city` on the same date.

In [ ]:
train_merged = train_merged.merge(
    local_holidays,
    left_on=["date", "city"],
    right_on=["date", "locale_name"],
    how="left",
    validate="many_to_one"
)

test_merged = test_merged.merge(
    local_holidays,
    left_on=["date", "city"],
    right_on=["date", "locale_name"],
    how="left",
    validate="many_to_one"
)

train_merged = train_merged.drop(columns=["locale_name"])
test_merged = test_merged.drop(columns=["locale_name"])

train_merged["local_holiday"] = (
    train_merged["local_holiday"].fillna(0).astype(int)
)

test_merged["local_holiday"] = (
    test_merged["local_holiday"].fillna(0).astype(int)
)

print("Local holidays in the training set:")
print(train_merged["local_holiday"].value_counts())

print("\nLocal holidays in the test set:")
print(test_merged["local_holiday"].value_counts())

### **Integrating Local Holidays**

The `local_holiday` indicator is 1 when a store's city has a local holiday on that date and 0 otherwise. The notebook reports 11,880 matching training rows and 66 matching test rows; the test period includes local holidays even though it has no regional-holiday matches.

Join on `date` and store `city` matched to holiday `locale_name`, using a left join and `validate="many_to_one"`. Drop the redundant `locale_name` column after the join, then fill unmatched indicators with 0 and convert them to integers.

In [ ]:
# 1. Merge national holidays missing from the merged tables
train_merged = train_merged.drop(columns=["national_holiday"], errors="ignore")
test_merged = test_merged.drop(columns=["national_holiday"], errors="ignore")

train_merged = train_merged.merge(
    national_holidays,
    on="date",
    how="left",
    validate="many_to_one"
)

test_merged = test_merged.merge(
    national_holidays,
    on="date",
    how="left",
    validate="many_to_one"
)

train_merged["national_holiday"] = train_merged["national_holiday"].fillna(0).astype(int)
test_merged["national_holiday"] = test_merged["national_holiday"].fillna(0).astype(int)

# 2. Check all holiday features in the final tables
print("Training:", train_merged.shape)
print("Test:", test_merged.shape)

print("\nMissing values:")
print(train_merged[
    ["national_holiday", "regional_holiday", "local_holiday"]
].isna().sum())

# 3. Detailed counts of non-holidays (0) and holidays (1)
print("\n=== REGULAR DAYS (0) vs HOLIDAYS (1) IN TRAINING SET ===")
for col in ["national_holiday", "regional_holiday", "local_holiday"]:
    print(f"\nDistribution of {col}:")
    print(train_merged[col].value_counts())

print("\n=== REGULAR DAYS (0) vs HOLIDAYS (1) IN TEST SET ===")
for col in ["national_holiday", "regional_holiday", "local_holiday"]:
    print(f"\nDistribution of {col}:")
    print(test_merged[col].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for a combined plot
# Melt the holiday columns into long format
df_melted = train_merged.melt(
    value_vars=['national_holiday', 'regional_holiday', 'local_holiday'],
    var_name='Holiday Type',
    value_name='Status'
)

# Map 0 and 1 to readable labels
df_melted['Status'] = df_melted['Status'].map({0: 'Regular Day (0)', 1: 'Holiday (1)'})

# Count the frequency of each combination
df_counts = df_melted.groupby(['Holiday Type', 'Status']).size().reset_index(name='Total Days')

# Configure the visualization
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

# Create a grouped bar chart
ax = sns.barplot(
    data=df_counts,
    x='Holiday Type',
    y='Total Days',
    hue='Status',
    palette=['#34495e', '#e74c3c']
)

# Add clear titles and formatting
plt.title("Regular Days vs Holidays in Training Set", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Holiday Category", fontsize=12)
plt.ylabel("Absolute Number of Records", fontsize=12)
plt.yscale('log') # Log scale to make the large difference in counts easier to see

# Add values above the bars for clarity
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(
            p.get_x() + p.get_width() / 2.0,
            height * 1.15, # Vertical position adjustment for the logarithmic scale
            f'{int(height):,}',
            ha='center',
            fontweight='bold',
            fontsize=10
        )

plt.tight_layout()
plt.show()

In [ ]:
print("Training:", train_merged.shape)
print("Test:", test_merged.shape)

print("\nMissing values:")
print(train_merged[
    ["national_holiday", "regional_holiday", "local_holiday"]
].isna().sum())

print("\nHoliday totals:")
print(train_merged[
    ["national_holiday", "regional_holiday", "local_holiday"]
].sum())

print(test_merged[
    ["national_holiday", "regional_holiday", "local_holiday"]
].sum())

In [ ]:
train_merged.groupby(
    ["store_nbr", "family"]
)["date"].count().value_counts().head()

The training data contains 1,782 store-family series, each with 1,684 original records. Every combination has the same number of observations, but that alone does not guarantee a complete calendar.

In [ ]:
train_merged.groupby(
    ["store_nbr", "family"]
)["date"].apply(
    lambda x: x.sort_values().diff().dt.days.ne(1).sum()
).sum()

There are 8,910 adjacent-record intervals that are not exactly one day apart. Some may indicate missing dates; the first observation in each series must also be considered when interpreting date differences.

In [ ]:
intervalos = (
    train_merged.groupby(["store_nbr", "family"])["date"]
    .diff().dt.days
)

intervalos.value_counts().sort_index()

The interval check found:

- 2,991,978 intervals of one day;
- 7,128 intervals of two days, indicating a missing day.

Because `shift(1)` and `shift(7)` count rows, a gap can make them differ from one and seven calendar days.

In [ ]:
(intervalos == 2).groupby(
    [train_merged["store_nbr"], train_merged["family"]]
).sum().value_counts().sort_index()

Each of the 1,782 store-family series has exactly four two-day gaps. These gaps must be handled before calculating lags so that a seven-row shift corresponds to seven calendar days.

In [ ]:
dates = pd.date_range(
    start=train_merged["date"].min(),
    end=train_merged["date"].max(),
    freq="D"
)

missing_dates = dates.difference(
    train_merged["date"].unique()
)

missing_dates

The missing dates are December 25 in 2013, 2014, 2015, and 2016. Since all series omit these dates, the project assumes stores were closed and assigns zero only to `sales` on inserted rows. Validate this assumption; other columns are not filled automatically. Reindex before creating lags so `shift(7)` represents seven consecutive calendar days.

In [ ]:
train_merged[
    train_merged["date"].isin(missing_dates)
].shape

Insert the missing dates before creating temporal features. Under the documented Christmas-closure assumption, set only `sales` to zero on the new rows. Leave promotions, oil prices, identifiers, and other attributes missing unless a column-specific rule justifies filling them.

In [ ]:
index = pd.MultiIndex.from_product(
    [
        train_merged["store_nbr"].unique(),
        train_merged["family"].unique(),
        dates
    ],
    names=["store_nbr", "family", "date"]
)

print("Expected combinations:", len(index))
print("Current rows:", len(train_merged))

Reindexing adds 7,128 rows:

- Expected total: 3,008,016
- Original total: 3,000,888
- Added rows: 7,128 (four dates × 1,782 series)

The next step aligns the daily index while preserving existing columns.

In [ ]:
from src.data_cleaning import reindex_daily_sales

rows_before = len(train_merged)
train_merged = reindex_daily_sales(
    train_merged,
    key_columns=("store_nbr", "family"),
    date_column="date",
    sales_column="sales",
    missing_sales_value=0.0,
)

print("Shape after reindexing:", train_merged.shape)
print("Rows inserted:", len(train_merged) - rows_before)
print("Missing sales after explicit imputation:", train_merged["sales"].isna().sum())


The daily index now contains all expected store-family dates. Next, inspect which columns are missing values on the 7,128 inserted rows and handle each according to its meaning.

In [ ]:
train_merged.isna().sum()[lambda x: x > 0]

# **Feature Engineering**

In [ ]:
from src.feature_engineering import engineer_features

train_merged = engineer_features(train_merged)
test_merged = engineer_features(test_merged)

print(train_merged[
    ["date", "year", "month", "day", "day_of_week", "weekend"]
].head())

print("\nTest:")
print(test_merged[
    ["date", "year", "month", "day", "day_of_week", "weekend"]
].head())


### **Creating Calendar Features**

Calendar features are extracted from `date` so a later model can represent recurring patterns by year, month, day, and weekday. The notebook applies the same transformations to the merged training and test tables:

```python
for df in [train_merged, test_merged]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek
    df["weekend"] = (df["day_of_week"] >= 5).astype(int)
```

The `.dt` accessor exposes datetime components. Pandas numbers weekdays from Monday (`0`) to Sunday (`6`); therefore, values greater than or equal to 5 identify Saturdays and Sundays. Converting the boolean result with `.astype(int)` produces a 1/0 weekend indicator.

# **Create a promotion feature**

In [ ]:
print(train_merged[["onpromotion", "has_promotion"]].head(10))

print("\nTraining distribution:")
print(train_merged["has_promotion"].value_counts(dropna=False))

print("\nTest distribution:")
print(test_merged["has_promotion"].value_counts(dropna=False))


### **Promotion Indicator (`has_promotion`)**

The binary `has_promotion` feature indicates whether `onpromotion` is greater than zero. Observed zero-promotion values become 0, positive values become 1, and missing promotion data remains missing rather than being interpreted as no promotion.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate training-set counts and proportions
train_counts = train_merged['has_promotion'].value_counts()
train_pct = train_merged['has_promotion'].value_counts(normalize=True) * 100

# Calculate test-set counts and proportions
test_counts = test_merged['has_promotion'].value_counts()
test_pct = test_merged['has_promotion'].value_counts(normalize=True) * 100

# Build a consolidated DataFrame for plotting
df_plot = pd.DataFrame({
    'Dataset': ['Training', 'Training', 'Test', 'Test'],
    'Promotion Status': ['No Promotion (0)', 'With Promotion (1)', 'No Promotion (0)', 'With Promotion (1)'],
    'Records': [train_counts[0], train_counts[1], test_counts[0], test_counts[1]],
    'Porcentagem': [train_pct[0], train_pct[1], test_pct[0], test_pct[1]]
})

# Configure chart style and size
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Absolute counts on a logarithmic scale
ax1 = sns.barplot(
    data=df_plot,
    x='Dataset',
    y='Records',
    hue='Promotion Status',
    palette=['#34495e', '#e74c3c'],
    ax=axes[0]
)
axes[0].set_yscale('log')  # Log scale to handle the difference in data volume
axes[0].set_title("Total Number of Records (Log Scale)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Number of Records")
axes[0].set_xlabel("Dataset")

for p in ax1.patches:
    height = p.get_height()
    if height > 0:
        ax1.text(
            p.get_x() + p.get_width() / 2.0,
            height * 1.15,
            f'{int(height):,}',
            ha='center',
            fontweight='bold',
            fontsize=10
        )

# Chart 2: Percentage distribution (%)
ax2 = sns.barplot(
    data=df_plot,
    x='Dataset',
    y='Porcentagem',
    hue='Promotion Status',
    palette=['#34495e', '#e74c3c'],
    ax=axes[1]
)
axes[1].set_title("Relative Proportion (%)", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Porcentagem (%)")
axes[1].set_xlabel("Dataset")
axes[1].set_ylim(0, 110)

for p in ax2.patches:
    height = p.get_height()
    if height > 0:
        ax2.text(
            p.get_x() + p.get_width() / 2.0,
            height + 2,
            f'{height:.1f}%',
            ha='center',
            fontweight='bold',
            fontsize=10
        )

plt.suptitle("Promotions: Training vs Test", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# **Prepare the sales history**

In [ ]:
train_merged = train_merged.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

print(train_merged[
    ["date", "store_nbr", "family", "sales"]
].head(10))

### **Sorting the Sales History**

Before creating lag or rolling features, sort the data by `store_nbr`, `family`, and `date`. Reset the index with `drop=True` to create a clean sequential index without preserving the old index as an extra column.

# **Previous-day sales**

In [ ]:
print(train_merged[["date", "store_nbr", "family", "sales", "sales_lag_1"]].head(10))


### **One-Day Sales Lag (`sales_lag_1`)**

```python
train_merged["sales_lag_1"] = (
    train_merged.groupby(["store_nbr", "family"])["sales"].shift(1)
)
```

Grouping isolates the sales history for each store-family pair. `shift(1)` places the previous row's sales in the current row, so on a complete daily series it represents the previous day. The first row in each series has no prior observation and therefore remains `NaN`.

# **Sales from seven days earlier**

In [ ]:
print(train_merged[["date", "sales", "sales_lag_1", "sales_lag_7"]].head(12))


# **Seven-day moving average**

In [ ]:
print(train_merged[["date", "sales", "sales_lag_1", "sales_rolling_7"]].head(12))


### **Seven-Day Lag and Moving Average**

`sales_lag_7` retrieves sales from seven rows earlier within each store-family series. On a complete daily calendar, this is the same weekday from the previous week; the first seven rows have no prior value.

`sales_rolling_7` summarizes the previous seven completed days:

```python
train_merged["sales_rolling_7"] = (
    train_merged.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda values: values.shift(1).rolling(7).mean())
)
```

`shift(1)` excludes the current day's sales to prevent target leakage, `rolling(7)` selects the seven previous rows, and `mean()` calculates their average. With a complete series, the first value appears on January 8, 2013, and is the average of January 1–7 (15 / 7 ≈ 2.14). The next value averages January 2–8 (17 / 7 ≈ 2.43).

The one-day lag describes recent sales, the seven-day lag captures weekly seasonality, and the moving average smooths short-term fluctuations.

# **30-day moving average**

In [ ]:
print(train_merged[["date", "sales", "sales_rolling_7", "sales_rolling_30"]].head(35))


The 30-day window is created by `rolling(30)`, not by `shift(1)`. The shift moves the series back one row to exclude the current day's target; `rolling(30)` then calculates a window over the 30 preceding rows.

To visualize how the 30-day window moves, compare the windows used on two consecutive dates. The first window contains the 30 completed days before the first date. On the next date, the window drops the oldest observation and includes the newly completed day. At least 30 prior observations are required before the moving average is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Filter one store-family series for a clear, uncluttered view
example_plot = train_merged[
    (train_merged['store_nbr'] == 1) &
    (train_merged['family'] == 'AUTOMOTIVE') &
    (train_merged['date'].between('2013-01-01', '2013-05-31'))
]

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Chart 1: Sales and moving averages over time
axes[0].plot(example_plot['date'], example_plot['sales'], label='Actual Sales (sales)', color='#3498db', alpha=0.4, linewidth=1.5)
axes[0].plot(example_plot['date'], example_plot['sales_rolling_7'], label='7-Day Moving Average (sales_rolling_7)', color='#e67e22', linewidth=2)
axes[0].plot(example_plot['date'], example_plot['sales_rolling_30'], label='30-Day Moving Average (sales_rolling_30)', color='#2ecc71', linewidth=3)
axes[0].set_title("Smoothing and Trend: Daily Sales vs Moving Averages (Store 1 - AUTOMOTIVE)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Data", fontsize=12)
axes[0].set_ylabel("Units Sold", fontsize=12)
axes[0].legend(fontsize=11, loc='upper left')

# Chart 2: Correlations between generated temporal features
analysis_columns = ['sales', 'sales_lag_1', 'sales_lag_7', 'sales_rolling_7', 'sales_rolling_30']
corr_matrix = train_merged[analysis_columns].corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='Blues',
    fmt='.3f',
    linewidths=1,
    ax=axes[1],
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
axes[1].set_title("Association (Correlation) Between Temporal Features in the Full Dataset", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Aggregate data efficiently for the bar-chart analysis
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Map weekdays to readable names
weekday_names = {
    0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'
}
train_plot_bars = train_merged.copy()
train_plot_bars['day_name'] = train_plot_bars['day_of_week'].map(weekday_names)

# 1. Bar chart: Average sales by weekday
sns.barplot(
    data=train_plot_bars,
    x='day_name',
    y='sales',
    order=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    ax=axes[0],
    palette='Blues_d',
    errorbar=None
)
axes[0].set_title("Average Sales by Weekday", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Weekday")
axes[0].set_ylabel("Average Sales (units)")

# 2. Bar chart: Average sales on weekends versus weekdays
sns.barplot(
    data=train_plot_bars,
    x='weekend',
    y='sales',
    ax=axes[1],
    palette='Oranges_d',
    errorbar=None
)
axes[1].set_title("Average Sales: Weekend vs Weekday", fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Weekday (0)', 'Weekend (1)'])
axes[1].set_xlabel("Day Type")
axes[1].set_ylabel("Average Sales (units)")

# 3. Bar chart: Average sales with versus without promotion
sns.barplot(
    data=train_plot_bars,
    x='has_promotion',
    y='sales',
    ax=axes[2],
    palette='Reds_d',
    errorbar=None
)
axes[2].set_title("Average Sales: With vs Without Promotion", fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['No Promotion (0)', 'With Promotion (1)'])
axes[2].set_xlabel("Promotion Status")
axes[2].set_ylabel("Average Sales (units)")

plt.suptitle("Sales Patterns by Key Features (Bar Charts)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### **Calculating the 30-Day Moving Average (`sales_rolling_30`)**

The 30-day feature follows the same leakage-safe pattern as the seven-day average:

```python
train_merged["sales_rolling_30"] = (
    train_merged.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda values: values.shift(1).rolling(30, min_periods=30).mean())
)
```

The grouped calculation prevents information from one store-family series from entering another. `shift(1)` excludes the target day's sales; `rolling(30, min_periods=30)` requires 30 complete prior observations. Therefore, the first 30 rows in each series remain missing. On a complete daily calendar, the value for a date is the mean of the 30 days immediately before it.

# **30-day moving average for the training set**

In [ ]:
print(train_merged[["date", "sales", "sales_rolling_7", "sales_rolling_30"]].head(35))


In [ ]:
print("Missing values in these columns")
train_merged[
    ["sales_lag_1", "sales_lag_7", "sales_rolling_7", "sales_rolling_30"]
].isna().sum()

After reindexing, each store-family combination has 1,688 daily dates: 1,684 original observations plus four Christmas dates. Initial lag and moving-average values remain missing because each feature requires enough prior history.

**Organization note:** the gap diagnosis and daily reindexing were moved before temporal feature engineering in this project copy. This ensures that row-based shifts represent calendar days and that inserted Christmas dates are considered when calculating the history.

The reindexing step adds 7,128 rows across 1,782 store-family series. Only `sales` on those new dates is set to zero under the documented store-closure assumption; other fields remain missing unless a separate rule is defined. Calendar, promotion, lag, and rolling features are calculated after reindexing. Lags are grouped by store and family, and moving averages exclude the current day's sales. Their initial missing values are expected because there is not enough prior history.